In [4]:
import ipyleaflet
import ipywidgets
import geopandas as gpd
from raster_tools import Raster
from PIL import Image as PILimage
import numpy as np, base64, io, os, requests
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def create_download_link(filename, title = "Click here to download: "):  
    fl = open(filename, "rb")
    data = fl.read()
    b64 = base64.b64encode(data)
    payload = b64.decode()
    fl.close()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank" style="color:blue">{title}</a>'
    html = html.format(payload=payload,title=title+f' {filename}',filename=filename[5:])
    return html

def _get_database_from_repo():
    repo_owner = "jshogland"
    repo_name = "RPMS"
    branch = "main"
    fnm="./images/RPMS_summary.gpkg"
    if not os.path.exists(fnm):
        file_path = "images/"+fnm
        url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/{branch}/{file_path}"

        # Download the file
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # raise error for bad responses

        # Save the file locally
        with open(fnm, "wb") as f:
            f.write(response.content)
    return fnm

def _get_file_from_repo():
    global pid
    repo_owner = "jshogland"
    repo_name = "RPMS"
    branch = "main"
    fnm="./images/ppa_tile_"+pid+'.tif'
    if not os.path.exists(fnm):
        file_path = "images/"+fnm
        url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/{branch}/{file_path}"

        # Download the file
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # raise error for bad responses

        # Save the file locally
        with open(fnm, "wb") as f:
            f.write(response.content)

    return fnm

def _update_img():
    global gdf, pid, lbl, lbl2, img
    lbl.value = 'Polygon:' + pid
    lbl2.value= 'Preview lbs raster'
    flnm=_get_file_from_repo()
    #flnm='./ppa/ppa_tile_'+pid+'.tif'
    ax=gdf[gdf.poly_index==int(pid)].plot(facecolor='none',edgecolor='#069AF3',zorder=1,figsize=(8,5),linewidth=2)
    ax=Raster(flnm).plot(cmap='viridis',robust=True, ax=ax,zorder=0)
    ax.axes.set_title('')
    plt.title=""
    buf=io.BytesIO()
    ax.figure.savefig(buf,format='png',bbox_inches='tight')
    plt.close(ax.figure)
    buf.seek(0)
    pil_img = PILimage.open(buf)
    img_bytes=io.BytesIO()
    pil_img.save(img_bytes,format='PNG')
    img_bytes.seek(0)
    img.value=img_bytes.read()
    img.format='png'

def _add_attributes():
    global gdf, pid, outwdg
    tdf=gdf[gdf.poly_index==int(pid)]
    with outwdg:
        clear_output()
        print("Row Attributes:")
        display(tdf.T)


def handle_click(event, feature, **kwargs):
    """
    event: 'click'
    feature: dict with GeoJSON feature properties
    """
    props = feature['properties']

    global pid
    pid=str(props['poly_index'])
    flnm='./images/ppa_tile_'+pid+'.tif'
    _update_img()
    html.value=create_download_link(flnm)
    _add_attributes()
    

#create left panel
pid=''
if not os.path.exists('./images'): os.mkdir('./image')
vbox_layout=ipywidgets.Layout(width='25%',border='solid 1px black')

lblt=ipywidgets.HTML(value="<b style='font-size:20px;'>RPMS Dashboard - Click on a allotment to preview lbs raster surface and attributes</b>")

lbl=ipywidgets.Label('Polygon: '+str(pid),)
lbl2=ipywidgets.Label('')
img=ipywidgets.Image()
html=ipywidgets.HTML("")
outwdg=ipywidgets.Output()
lb=ipywidgets.VBox([lbl,lbl2,img,html,outwdg],layout=vbox_layout)

#create right panel (the map)
lc = ipyleaflet.LayersControl()
lc.position='topright'
dr=ipyleaflet.DrawControl()
dr.position='topleft'
bmaps=ipyleaflet.basemaps
m=ipyleaflet.Map(controls=[lc,dr],scroll_wheel_zoom=True,center=(41,-119),zoom=5,layout=ipywidgets.Layout(height='90%'))

tly1=ipyleaflet.basemap_to_tiles(bmaps.Esri.WorldImagery)
tly1.base = False
tly1.name = 'World Imagery'

#get polygons
gpk_path=_get_database_from_repo()
lynm='allot_simp'
gdf=gpd.read_file(gpk_path,layer=lynm)
geo_data=ipyleaflet.GeoData(geo_dataframe=gdf,name='Allotments',hover_style={'fillColor':'red','fillOpacity':0.2})


# Attach click event to map
geo_data.on_click(handle_click)

# Add layers to the map
m.add(tly1)
m.add(geo_data)
rb=ipywidgets.VBox([m],layout=ipywidgets.Layout(width='75%',height='700px'))

#Create the view widget
vw=ipywidgets.HBox([lb,rb])
view=ipywidgets.VBox([lblt,vw])
display(view)

In [3]:
import requests
from pathlib import Path

# Replace with your repo, branch, and file path
repo_owner = "jshogland"
repo_name = "RPMS"
branch = "main"
file_path = "images/ppa_tile_1.tif"
outfile=r"C:\Users\jshogland\John\temp\ppa_tile_1.tif"

# Build the raw GitHub URL
url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/{branch}/{file_path}"

# Download the file
response = requests.get(url, timeout=10)
response.raise_for_status()  # raise error for bad responses

# Save the file locally
with open(outfile, "wb") as f:
    f.write(response.content)

print(f"Downloaded {file_path} successfully.")

Downloaded images/ppa_tile_1.tif successfully.
